# TIL — 2026-11-13

> 오늘은 두 가지를 했다. ① Chapter 09 React Mini Pop-up Store STEP 20~23(장바구니 State·담기·수량 ±·총액) ② 나만의 AI 비서 Lesson 01(첫 실행, Mock 모드)

## 오늘 한 일

| 구분 | 내용 | 상태 |
| --- | --- | --- |
| Pop-up Store STEP 20 | `CartItem` 타입 + `useState<CartItem[]>([])` | 완료 |
| Pop-up Store STEP 21 | 담기 Event, 불변 업데이트, 재고 제한 | 완료 (Playwright Smoke ② 통과) |
| Pop-up Store STEP 22 | 수량 + / -, 1에서 - 누르면 제거 | 완료 (Playwright로 확인) |
| Pop-up Store STEP 23 | Derived State로 총 수량·총액 계산 | 완료 (Playwright로 확인) |
| 환경 | Playwright + Chromium 설치 (scratchpad) | 완료 |
| AI 비서 Lesson 01 | 가상환경, Mock 실행, `main.py` 구조 읽기, `.gitignore` 확인 | 완료 |

---

# Part 1. Chapter 09 — React Mini Pop-up Store (STEP 20~23)

## 1. cart State 만들기 (STEP 20)

```ts
// types/cart.ts
export type CartItem = Product & { quantity: number };

// app/store/page.tsx (Client Component)
const [cart, setCart] = useState<CartItem[]>([]);
```

- 기존 `Product` 타입을 재사용하고 `quantity`만 더한 타입.
- 장바구니는 서버 DB가 아니라 **브라우저 컴포넌트 State**에 있음. 이 단계에서는 서버 저장·주문 POST 없음.
- State는 상위 컴포넌트(`/store` 페이지) **한 곳에서만** 소유. `ProductCard`는 State 없이 `onAdd` 이벤트만 부모로 전달.
- `useState`는 클라이언트 훅이라 `"use client"` 경계 안에서만 사용 가능.

## 2. 담기 Event와 불변 업데이트 (STEP 21)

- 변경 로직을 `lib/cart.ts`의 순수 함수로 분리(`addToCart` 등). 모두 **기존 배열·객체를 수정하지 않고 새 배열을 반환**.
- 규칙:
  - 같은 `product.id`면 새 줄을 만들지 않고 `quantity`만 +1
  - 처음 담으면 `quantity = 1`로 추가
  - `stock <= 0`(품절)이거나 `quantity >= stock`이면 변화 없음
- 페이지에서는 `setCart((prev) => addToCart(prev, product))` 형태의 **함수형 업데이트**로 연결.
  - `cart.push()` 같은 직접 수정은 React가 변화를 못 알아챌 수 있음.
- Badge는 줄 수가 아니라 **quantity 합계**.

## 3. 수량 + / - (STEP 22)

- `increaseQuantity`: `stock` 이하까지만 증가. 재고 상한이면 `+` 버튼도 `disabled`.
- `decreaseQuantity`: `quantity`가 1일 때 `-`를 누르면 **항목 제거**. 0이나 음수는 cart에 남지 않음. `flatMap`으로 "그대로 / 수정 / 제거"를 한 번에 처리.
- 항목은 상품 `id`로 식별 → 한 상품의 변경이 다른 상품 `quantity`에 영향 없음.

## 4. Derived State로 총액 계산 (STEP 23)

```ts
const totalCount = cart.reduce((sum, item) => sum + item.quantity, 0);
const totalPrice = cart.reduce((sum, item) => sum + item.price * item.quantity, 0);
```

- `cart`로부터 계산할 수 있는 값은 **별도 `useState`로 중복 저장하지 않음**. 렌더할 때마다 계산하면 항상 `cart`와 일치.
- Badge, "총 수량 N개", "합계 N원"이 모두 같은 `cart`에서 나옴. 금액은 정수 원 단위, 표시는 `toLocaleString("ko-KR")`.
- **기록할 점**: 화면의 총액은 표시용일 뿐. 브라우저 값은 조작될 수 있으므로 **최종 주문 합계는 Backend가 DB 가격으로 다시 계산**해야 함. (주문 POST는 아직 미구현)

## 5. 실제 브라우저로 검증 (Playwright)

STEP 19에서 HOLD였던 Playwright를 오늘 설치해서 실제 Chromium으로 확인.

| 동작 | Badge / 수량 | 합계 |
| --- | --- | --- |
| 초기 | 0 | 0원 |
| 담기 → Badge | 0 → 1 (Smoke ②) | |
| 같은 상품 2회 | 2 (줄 1개, quantity 2) | |
| 쿠키 2개 + 아메리카노 1개 | 3 | **8,500원** (강의 기대값과 일치) |
| 아메리카노 + | 4 | 11,000원 |
| 쿠키 - | 3 | 8,000원 |
| 아메리카노 제거 | 1 | 3,000원 |
| 전부 제거 | 0 | 0원 |
| 딸기 케이크(재고 8) 12번 클릭 | 8에서 멈춤, + 비활성 | |

- 브라우저 Console 오류·key 경고: 없음 (어제 TODO였던 항목 해소)
- 쿠키 수량 1에서 `-` → 줄 사라짐, 다른 항목 수량은 그대로
- 순수 함수는 별도로 Node 스크립트로도 확인(원본 배열 불변, 재고 상한, 품절, 제거).
- `tsc --noEmit`, `eslint` 통과.

## 6. 환경 메모

- Playwright는 프로젝트 `package.json`을 건드리지 않으려고 **scratchpad에 따로 설치**. 브라우저는 `%LOCALAPPDATA%\ms-playwright`에 설치됨. 강의에서 스크립트 파일 제출이 필요하면 프로젝트로 옮기면 됨.
- 테스트 스크립트는 `data-testid`(`cart-badge`, `qty-{id}`, `total-count`, `total-price`)와 버튼의 `aria-label`로 요소를 찾음.
- `node --experimental-strip-types`로 `.ts` 파일을 바로 실행할 때 Windows 경로는 `file:///C:/...` URL로 import해야 함.

## 헷갈렸던 점 / 배운 점

- "**State는 최소로, 나머지는 계산**" — 총액·Badge를 따로 저장하면 cart와 어긋날 위험이 생김.
- 불변 업데이트는 `{ ...item, quantity: item.quantity + 1 }`처럼 **바꿀 것만 새 객체로 만들고 나머지는 그대로 유지**하는 것.
- 서버 HTML 확인(어제)과 실제 브라우저 클릭 확인(오늘)은 증거의 종류가 다름. 클릭 이후 상태는 브라우저에서만 확인 가능.

---

# Part 2. 나만의 AI 비서 — Lesson 01 첫 실행

## 오늘 한 일
- Python 가상환경(`.venv`)을 만들고 `requirements.txt` 패키지를 설치했다.
- `main.py`를 API 키 없이 Mock 모드로 실행해 봤다.
- `main.py`의 구조를 읽고 이해했다.
- `MY_ASSISTANT_GOAL.md`가 git에서 무시되는지 확인했다.

## 1. 가상환경 만들기
```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```
- 처음에는 프로젝트 루트에 만들었다가, 실습 폴더(`practice/my-ai-assistant`) 아래에 다시 만들었다.
- 설치 끝에 빨간 글씨가 나왔지만 pip 업그레이드 안내일 뿐이었다. 오류가 아니었다.
- `.venv/`는 `.gitignore`에 들어 있어서 git에 올라가지 않는다.
- 활성화: `.\.venv\Scripts\Activate.ps1`

## 2. Mock 모드 실행
```powershell
.\.venv\Scripts\python.exe main.py "이번 주 과제를 정리해 줘"
```
- 결과: `[MOCK 예시] 질문을 받았습니다: ...`
- Mock은 질문을 그대로 되돌려 줄 뿐이다. 질문을 이해하거나 답하지는 못한다.
- 빈 질문(`" "`)을 넣으면 `ERROR: 질문을 입력해야 합니다.`가 나오고 종료 코드는 2다.

## 3. main.py 구조 정리
| 궁금한 것 | 관련 코드 |
|---|---|
| 질문이 들어오는 곳 | `main()`의 `argparse` → `args.question` |
| Mock이 하는 일 | `MockClient.ask()`: 공백 제거 → 빈 값 검사 → 고정 문장 반환 |
| 답변 데이터 | `Reply`: text, model, 토큰 수 등을 담는 dataclass(`frozen=True`) |
| mock vs live | `make_client()`가 `MockClient` 또는 `OpenAIClient`/`GeminiClient`를 선택 |
| 빈 질문 처리 | `ask()`가 `ValueError` 발생 → `main()`의 `except`가 받아 `ERROR:` 출력, `return 2` |

- `--mode`의 기본값은 `mock`이다. live 모드는 `.env`의 API 키가 있어야 하고 비용이 들 수 있다.
- 오류를 던지는(`raise`) 곳과 처리하는(`except`) 곳이 다르다는 걸 알게 됐다.

## 4. 알게 된 것들
- `Get-Content`는 설치 파일이 아니라 PowerShell 내장 명령(cmdlet)이다. 소속 모듈은 `Microsoft.PowerShell.Management`다.
- `Get-Content .`는 `.`가 폴더라서 오류가 난다. 파일을 지정해야 한다.
- `git check-ignore -v 파일명`으로 어떤 규칙 때문에 무시되는지 확인할 수 있다. `MY_ASSISTANT_GOAL.md`는 이미 `.gitignore`에 있었다.

---

## 검증 상태

- ✅ STEP 20~23: `tsc --noEmit`, `eslint` 통과, 실제 Chromium(Playwright)에서 담기·±·제거·총액 확인, Console 오류·경고 없음
- ✅ 총액 8,500원 / 수량 3개가 강의 기대값과 일치
- ⬜ 주문 POST·Backend 재계산은 아직 구현 전 (이후 STEP)
- ⬜ AI 비서 Lesson 01은 직접 실행한 내용을 옮겨 적은 것(이 TIL에서 따로 재실행하지 않음)

## TODO

- [ ] STEP 24 이후 진행 (STEP 25에서 임시 상품 배열을 `GET /products` 응답으로 교체)
- [ ] 최종 주문 합계를 Backend가 DB 가격으로 재계산하는 단계에서 오늘 기록한 "표시용 총액 vs 서버 재계산" 다시 확인
- [ ] 어제 TODO 이어서: `/members` 다크 모드 제목 색, `member-next-app` 위치 정리, 루트 `.gitignore`, `assets/images/phase03/.env` 원본 정리
- [ ] AI 비서: `tests/test_main.py` 자동 검증 실행해 보기
- [ ] AI 비서: 다음 강의에서 실제 OpenAI API 연결하기 (live 모드, 비용 주의)
